In [ ]:
# K1

# Praise the Lord 🙏

# House Price - Advance Regression Techniques 

# Notebook Created on 26 October 2025

# Forked Code

# 26 October 2025

# import libraries 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor # Minimum for a modern benchmark

# --- Configuration (Minimal) ---

DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 # Standard seed for reproducibility

# --- 1. DATA LOADING & INITIAL PREP (Minimal Features) ---

try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    # df_test is not strictly needed for just a quick run/evaluation
except FileNotFoundError:
    print("FATAL ERROR: Files not found.")
    raise

# --- 2. CRITICAL 'TRIMMING' (Outlier Removal) ---
# Remove the most extreme outliers based on GrLivArea

df_train = df_train[df_train['GrLivArea'] < 4000].copy()
y = np.log(df_train[TARGET])
X = df_train.drop(TARGET, axis=1)

# --- 3. MINIMAL FEATURE SET ---
# Core features from the Grandmaster's list, prioritized for impact.

MINIMAL_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 
    'KitchenQual', 'YrSold' 
]
X = X[MINIMAL_FEATURES]


# --- 4. MINIMAL PREPROCESSING PIPELINE ---

# Identify feature types based on the minimal set

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Define the ColumnTransformer (Median for numeric, OneHot for categorical)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ]
)

# --- 5. SPLIT DATA ---

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Transform the data

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)


# --- 6. MODELING (Minimum Specs) ---

print("--- MODEL EVALUATION (Minimum Specs) ---")

# A. Linear Regression (No hyperparams to tune)

lm_model = LinearRegression()
lm_model.fit(X_train_processed, y_train)
lm_rmse = np.sqrt(mean_squared_error(y_val, lm_model.predict(X_val_processed)))
print(f"LM RMSE (log): {lm_rmse:.5f}")

# B. Random Forest (Minimum n_estimators)
# n_estimators=10 is the absolute minimum for a quick run.

rf_model = RandomForestRegressor(
    n_estimators=10, 
    max_depth=5, # Small depth for quick training
    random_state=RANDOM_STATE, 
    n_jobs=-1
)
rf_model.fit(X_train_processed, y_train)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_model.predict(X_val_processed)))
print(f"RF RMSE (log): {rf_rmse:.5f}")

# C. XGBoost (Minimum n_estimators)
# n_estimators=10 is the absolute minimum for a quick run.

xgb_model = XGBRegressor(
    n_estimators=10, 
    learning_rate=0.3, # Default/high learning rate for fast convergence
    random_state=RANDOM_STATE, 
    n_jobs=-1
)
xgb_model.fit(X_train_processed, y_train)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_model.predict(X_val_processed)))
print(f"XGB RMSE (log): {xgb_rmse:.5f}")

# --- 7. FINAL PREDICTION AND SUBMISSION PREP ---

# 1. Choose the best model (XGBoost is usually the best candidate)
FINAL_MODEL = xgb_model

# 2. Load the test data
try:
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv')
except FileNotFoundError:
    print("FATAL ERROR: Test file not found.")
    raise

# 3. Apply the same minimal feature selection and preprocessing

X_test = df_test.copy()
X_test = X_test[MINIMAL_FEATURES] # Use the same features

# Transform the test data using the *fitted* preprocessor

X_test_processed = preprocessor.transform(X_test)

# 4. Make predictions
# Predict the LOG of SalePrice

log_predictions = FINAL_MODEL.predict(X_test_processed)

# Convert predictions back to original price scale (antilog/exponentiate)

final_predictions = np.exp(log_predictions)



# This coded was incorporated in 26 October 2025 Submission 

# 5. Create the submission DataFrame

submission_df = pd.DataFrame({
    'Id': df_test['Id'], # Use the Id from the original test data
    'SalePrice': final_predictions
})

# 6. Save the submission file

submission_df.to_csv('submission.csv', index=False)

print("\nSuccessfully created submission.csv!")

# --- 7. FINAL PREDICTION AND SUBMISSION PREP ---

# 1. Choose the best model (XGBoost is usually the best candidate)
FINAL_MODEL = xgb_model

# 2. Load the test data
try:
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv')
except FileNotFoundError:
    print("FATAL ERROR: Test file not found.")
    raise

# 3. Apply the same minimal feature selection and preprocessing

X_test = df_test.copy()
X_test = X_test[MINIMAL_FEATURES] # Use the same features

# Transform the test data using the *fitted* preprocessor

X_test_processed = preprocessor.transform(X_test)

# 4. Make predictions
# Predict the LOG of SalePrice

log_predictions = FINAL_MODEL.predict(X_test_processed)

# Convert predictions back to original price scale (antilog/exponentiate)

final_predictions = np.exp(log_predictions)

# 5. Create the submission DataFrame

submission_df = pd.DataFrame({
    'Id': df_test['Id'], # Use the Id from the original test data
    'SalePrice': final_predictions
})

# 6. Save the submission file

submission_df.to_csv('submission.csv', index=False)

print("\nSuccessfully created submission.csv!")

♨️

26 Oct 2025

OutPut

--- MODEL EVALUATION (Minimum Specs) ---
LM RMSE (log): 0.17229
RF RMSE (log): 0.18970
XGB RMSE (log): 0.18744

Submission Stats :

Latest Score
0.17138 V3

Best Score
0.17138 V2

Daily Submissions
2 / 10 used

In [ ]:
# K1 A 

# revised Python code, incorporating the LightGBM mode

# Forked Code

# 27 October 2025

# import libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor # Minimum for a modern benchmark
# --- NEW LIGHTGBM IMPORTS ---
import lightgbm as lgb
from lightgbm import LGBMRegressor
# ----------------------------

# --- Configuration (Minimal) ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 # Standard seed for reproducibility

# --- 1. DATA LOADING & INITIAL PREP (Minimal Features) ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    # df_test is not strictly needed for just a quick run/evaluation
except FileNotFoundError:
    print("FATAL ERROR: Files not found.")
    raise

# --- 2. CRITICAL 'TRIMMING' (Outlier Removal) ---
# Remove the most extreme outliers based on GrLivArea
df_train = df_train[df_train['GrLivArea'] < 4000].copy()
y = np.log(df_train[TARGET])
X = df_train.drop(TARGET, axis=1)

# --- 3. MINIMAL FEATURE SET ---
# Core features from the Grandmaster's list, prioritized for impact.
MINIMAL_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF',
    'KitchenQual', 'YrSold'
]
X = X[MINIMAL_FEATURES]


# --- 4. MINIMAL PREPROCESSING PIPELINE ---

# Identify feature types based on the minimal set
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Define the ColumnTransformer (Median for numeric, OneHot for categorical)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ]
)

# --- 5. SPLIT DATA ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Transform the data
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)


# --- 6. MODELING (Mid-Level Specs: LightGBM) ---
print("--- MODEL EVALUATION (Mid-Level LGBM) ---")

# Define Hyperparameters (Better than minimal baseline)
LGBM_PARAMS = {
    'objective': 'regression_l1',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'random_state': RANDOM_STATE,
    'n_jobs': -1
}

lgbm_model = LGBMRegressor(**LGBM_PARAMS)

# Train the model with early stopping
lgbm_model.fit(
    X_train_processed, y_train,
    # The eval_set and eval_metric arguments enable early stopping
    eval_set=[(X_val_processed, y_val)],
    eval_metric='rmse',
    # Stop if the validation metric doesn't improve for 50 rounds
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    # Feature names are passed for better logging/inspection (optional but good practice)
    feature_name=preprocessor.get_feature_names_out().tolist()
)

# Predict and Evaluate
lgbm_preds = lgbm_model.predict(X_val_processed)
lgbm_rmse = np.sqrt(mean_squared_error(y_val, lgbm_preds))

print(f"LGBM RMSE (log): {lgbm_rmse:.5f}")
print(f"LGBM Trees Trained: {lgbm_model.best_iteration_}")

# Submission file Creation 

# --- 7. LOAD AND PREPROCESS TEST DATA ---

# Load the test data
try:
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv')
except FileNotFoundError:
    print("FATAL ERROR: Test file not found.")
    raise

# Select the same minimal features used for training
X_test = df_test[MINIMAL_FEATURES].copy()

# Apply the ALREADY FIT preprocessor to the test data
# Note: DO NOT call .fit_transform(), only .transform()
X_test_processed = preprocessor.transform(X_test)

# --- 8. PREDICTION AND INVERSE TRANSFORM ---

# Make predictions on the preprocessed test set
test_predictions_log = lgbm_model.predict(X_test_processed)

# Inverse transform predictions: e^x - 1, since the target was log(SalePrice)
test_predictions_price = np.expm1(test_predictions_log)


# --- 9. GENERATE SUBMISSION FILE ---

# Create the submission DataFrame
submission_df = pd.DataFrame({
    'Id': df_test['Id'], # Use the original Id column from the test data
    'SalePrice': test_predictions_price
})

# Save the DataFrame to a CSV file (index=False is CRITICAL for Kaggle)
submission_df.to_csv('submission.csv', index=False)

print("\n--- SUBMISSION GENERATED ---")
print(f"File 'submission.csv' created with {len(submission_df)} predictions.")